# 02 website integration tests

**Owner:** Thomas. Original notebook cells: 94.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
# THOMAS: Website backend integration testing

import time
import json
import pandas as pd

website_test_results = []


def run_website_test(
    test_id,
    description,
    function,
    expected_error=None
):

    started = time.perf_counter()

    details = None
    error = None
    passed = False

    try:

        details = function()

        passed = expected_error is None

        actual_result = "ALLOWED"

    except AgentError as exc:

        error = str(exc)

        actual_result = "DENIED"

        passed = (
            expected_error is not None
            and error == expected_error
        )

    except Exception as exc:

        actual_result = "ERROR"

        error = (
            f"{type(exc).__name__}: {str(exc)}"
        )

        passed = False

    website_test_results.append({
        "test_id": test_id,
        "description": description,
        "expected_result": (
            "DENIED"
            if expected_error is not None
            else "ALLOWED"
        ),
        "actual_result": actual_result,
        "passed": bool(passed),
        "seconds": round(
            time.perf_counter() - started, 3
        ),
        "error": error
    })

    print(
        test_id,
        "PASSED" if passed else "FAILED",
        flush=True
    )

    return details


# Verify dashboard records and role permissions
def check_dashboard(
    role,
    subject,
    allowed_workflows
):

    response = eye_style_dashboard(
        role,
        subject
    )

    assert response["success"] is True
    assert response["mode"] == "demo"
    assert response["record_count"] > 0

    assert response["record_count"] == len(
        response["records"]
    )

    actual_workflows = {
        record["workflow_type"]
        for record in response["records"]
    }

    assert actual_workflows == allowed_workflows

    assert all(
        validate_dashboard_output(record)["valid"]
        for record in response["records"]
    )

    # Verify valid JSON payload
    json.dumps(response, allow_nan=False)

    return response


# WEB-01: Owner dashboard
owner_dashboard = run_website_test(
    "WEB-01",
    "Owner can access all authorized demo workflows",
    lambda: check_dashboard(
        "owner",
        "demo-owner",
        {
            "Scheduling",
            "Insurance Verification",
            "Optical Orders"
        }
    )
)


# WEB-02: Receptionist dashboard
receptionist_dashboard = run_website_test(
    "WEB-02",
    "Receptionist can access assigned workflows",
    lambda: check_dashboard(
        "receptionist",
        "demo-receptionist",
        {
            "Scheduling",
            "Insurance Verification",
            "Optical Orders"
        }
    )
)


# WEB-03: Optician dashboard
optician_dashboard = run_website_test(
    "WEB-03",
    "Optician receives optical records only",
    lambda: check_dashboard(
        "optician",
        "demo-optician",
        {"Optical Orders"}
    )
)


# WEB-04: Scheduling AI chat
def test_scheduling_chat():

    response = eye_style_chat(
        question=(
            "Use GetNoShowRisk to review synthetic "
            "appointment 2. Return its model risk "
            "score on a scale from 0 to 1 and "
            "recommended staff follow-up. "
            "Do not interpret the score as a "
            "probability. Return a final answer "
            "after retrieving the risk score."
        ),
        workflow="Scheduling",
        authenticated_role="receptionist",
        authenticated_subject="demo-receptionist"
    )

    assert response["success"] is True
    assert response["answer"].strip()

    assert "GetNoShowRisk" in response["sources"]

    assert RISK_SCORE_NOTICE in response["notices"]

    assert response["mode"] == "demo"

    json.dumps(response, allow_nan=False)

    return response


chat_response = run_website_test(
    "WEB-04",
    "Receptionist can submit scheduling questions",
    test_scheduling_chat
)


# WEB-05: Unauthorized workflow access
run_website_test(
    "WEB-05",
    "Optician cannot access scheduling",
    lambda: eye_style_chat(
        question="Show scheduled appointments.",
        workflow="Scheduling",
        authenticated_role="optician",
        authenticated_subject="demo-optician"
    ),
    expected_error="Access to this workflow is denied."
)


# WEB-06: Unknown role
run_website_test(
    "WEB-06",
    "Unknown user roles are rejected",
    lambda: eye_style_dashboard(
        "administrator",
        "unknown-user"
    ),
    expected_error="Unauthorized user role."
)


# WEB-07: Missing identity
run_website_test(
    "WEB-07",
    "Missing user identity is rejected",
    lambda: eye_style_dashboard(
        "owner",
        ""
    ),
    expected_error="User identity is required."
)


# WEB-08: Empty question
run_website_test(
    "WEB-08",
    "Empty AI questions are rejected",
    lambda: eye_style_chat(
        question="",
        workflow="Scheduling",
        authenticated_role="owner",
        authenticated_subject="demo-owner"
    ),
    expected_error="Question cannot be empty."
)


# WEB-09: RAG knowledge assistant
# Reuse Daniel's existing evaluation question.
# No additional embedding model is initialized.

def test_rag_chat():

    question_record = next(
        item
        for item in RAG_EVAL_QUESTIONS
        if item["question_id"] == "RAG-01"
    )

    response = eye_style_chat(
        question=question_record["question"],
        workflow="RAG",
        authenticated_role="receptionist",
        authenticated_subject="demo-receptionist"
    )

    assert response["success"] is True

    assert response["workflow"] == "RAG"

    assert response["answer"].strip()

    assert response["mode"] == "demo"

    assert response["abstained"] is False, (
        "RAG unexpectedly abstained."
    )

    assert response["sources"], (
        "RAG did not return source references."
    )

    expected_source = question_record[
        "expected_source"
    ]

    expected_section = question_record[
        "expected_section"
    ]

    assert any(
        expected_source.casefold() in source.casefold()
        and expected_section.casefold() in source.casefold()
        for source in response["sources"]
    ), "Expected RAG source and section were not returned."

    assert expected_source.casefold() in (
        response["answer"].casefold()
    ), "Expected source was not cited in the answer."

    assert response["reports"] == []

    assert response["requires_staff_review"] is True

    assert "retrieved_chunks" not in response

    json.dumps(response, allow_nan=False)

    return response


rag_chat_response = run_website_test(
    "WEB-09",
    "RAG returns cited knowledge-base answers",
    test_rag_chat
)


# WEB-10: Oversized AI question
run_website_test(
    "WEB-10",
    "Oversized questions are rejected",
    lambda: eye_style_chat(
        question="A" * 4001,
        workflow="Scheduling",
        authenticated_role="owner",
        authenticated_subject="demo-owner"
    ),
    expected_error="Question exceeds length limit."
)


# WEB-11: Role-specific capabilities
# Verify RAG availability without expanding
# operational workflow permissions.

def test_capabilities():

    expected_workflows = {
        "owner": {
            "Scheduling",
            "Insurance Verification",
            "Optical Orders",
            "RAG"
        },
        "receptionist": {
            "Scheduling",
            "Insurance Verification",
            "Optical Orders",
            "RAG"
        },
        "optician": {
            "Optical Orders",
            "RAG"
        }
    }

    capability_results = {}

    for role, workflows in expected_workflows.items():

        response = eye_style_capabilities(
            role,
            f"demo-{role}"
        )

        assert response["success"] is True

        assert response["read_only"] is True

        assert response["mode"] == "demo"

        assert response["rag_available"] is True

        assert response[
            "live_crystal_connected"
        ] is False

        assert set(
            response["available_workflows"]
        ) == workflows

        assert response[
            "automatic_patient_contact"
        ] is False

        json.dumps(response, allow_nan=False)

        capability_results[role] = response

    return capability_results


capability_response = run_website_test(
    "WEB-11",
    "All staff roles have correct RAG and workflow permissions",
    test_capabilities
)


# WEB-12: Insurance AI chat
def test_insurance_chat():

    response = eye_style_chat(
        question=(
            "Call read_crystal using "
            "GetVerificationStatus with empty parameters. "
            "Identify unresolved insurance verification "
            "records and recommend staff review. "
            "After retrieving the records, "
            "return a final written answer."
        ),
        workflow="Insurance Verification",
        authenticated_role="receptionist",
        authenticated_subject="demo-receptionist"
    )

    assert response["success"] is True

    assert "GetVerificationStatus" in response["sources"]

    assert response["answer"].strip()

    json.dumps(response, allow_nan=False)

    return response


insurance_chat_response = run_website_test(
    "WEB-12",
    "Receptionist can submit insurance questions",
    test_insurance_chat
)


# WEB-13: Optical AI chat
def test_optical_chat():

    response = eye_style_chat(
        question=(
            "Review frame orders between "
            "2026-09-01 and 2026-09-30. "
            "Identify orders ready for dispensing "
            "and recommend staff follow-up."
        ),
        workflow="Optical Orders",
        authenticated_role="optician",
        authenticated_subject="demo-optician"
    )

    assert response["success"] is True

    assert "GetOfficeFrameOrders" in response["sources"]

    assert response["answer"].strip()

    json.dumps(response, allow_nan=False)

    return response


optical_chat_response = run_website_test(
    "WEB-13",
    "Optician can submit optical questions",
    test_optical_chat
)


# WEB-14: RAG insufficient-evidence handling
# Verify that unsupported questions do not
# produce invented knowledge-base answers.

def test_rag_abstention():

    question_record = next(
        item
        for item in RAG_EVAL_QUESTIONS
        if item["insufficient_evidence_expected"] is True
    )

    response = eye_style_chat(
        question=question_record["question"],
        workflow="RAG",
        authenticated_role="optician",
        authenticated_subject="demo-optician"
    )

    assert response["success"] is True

    assert response["workflow"] == "RAG"

    assert response["mode"] == "demo"

    assert response["abstained"] is True, (
        "RAG failed to abstain from an unsupported question."
    )

    assert response["sources"] == []

    assert ABSTENTION_PREFIX.lower() in (
        response["answer"].lower()
    )

    assert response["reports"] == []

    assert response["requires_staff_review"] is True

    assert "retrieved_chunks" not in response

    json.dumps(response, allow_nan=False)

    return response


rag_abstention_response = run_website_test(
    "WEB-14",
    "RAG abstains when approved evidence is unavailable",
    test_rag_abstention
)


# Generate website integration evaluation
website_integration_eval = pd.DataFrame(
    website_test_results
)

total_tests = int(
    len(website_integration_eval)
)

passed_tests = int(
    website_integration_eval["passed"].sum()
)

failed_tests = total_tests - passed_tests


# Summarize RAG-specific interface tests
rag_interface_tests = website_integration_eval.loc[
    website_integration_eval["test_id"].isin(
        {"WEB-09", "WEB-11", "WEB-14"}
    )
]

rag_tests_passed = int(
    rag_interface_tests["passed"].sum()
)


website_integration_summary = {
    "evaluation_mode": "Synthetic demo",
    "total_tests": total_tests,
    "passed_tests": passed_tests,
    "failed_tests": failed_tests,
    "pass_rate": (
        float(passed_tests / total_tests)
        if total_tests > 0 else None
    ),
    "all_tests_passed": bool(
        total_tests > 0
        and passed_tests == total_tests
    ),
    "rag_interface_tests": int(
        len(rag_interface_tests)
    ),
    "rag_interface_tests_passed": rag_tests_passed,
    "rag_integration_passed": bool(
        len(rag_interface_tests) == 3
        and rag_tests_passed == 3
    ),
    "firebase_authentication_tested": False,
    "http_endpoint_tested": False,
    "live_website_tested": False,
    "ai_answer_review_pending": True
}


display(website_integration_eval)

print("\n=== WEBSITE INTERFACE TEST SUMMARY ===")

print(json.dumps(
    website_integration_summary,
    indent=2
))


# Display sample responses for final review
sample_responses = {
    "OWNER DASHBOARD": owner_dashboard,
    "SCHEDULING RESPONSE": chat_response,
    "INSURANCE RESPONSE": insurance_chat_response,
    "OPTICAL RESPONSE": optical_chat_response,
    "RAG KNOWLEDGE RESPONSE": rag_chat_response,
    "RAG ABSTENTION RESPONSE": rag_abstention_response,
    "ROLE CAPABILITIES": capability_response
}


for name, response in sample_responses.items():

    if response is not None:

        print("\n=== " + name + " ===")

        print(json.dumps(
            response,
            indent=2,
            allow_nan=False
        ))


# Display failed tests
if failed_tests > 0:

    print("\n=== FAILED WEBSITE TESTS ===")

    display(
        website_integration_eval.loc[
            website_integration_eval[
                "passed"
            ].eq(False)
        ]
    )

else:

    print(
        "\nAll internal website interface tests passed."
    )


if website_integration_summary[
    "rag_integration_passed"
]:

    print(
        "RAG website interface tests passed."
    )

else:

    print(
        "RAG website interface requires further review."
    )


print("\nAI-generated answers require final review.")
print("HTTP endpoint deployment remains pending.")
print("Firebase authentication remains pending.")
print("\nWebsite interface evaluation completed.")